In [ ]:
%load_ext sql

In [ ]:
# %sql sqlite:///:memory:

In [1]:
%load_ext sql
%sql sqlite:///my_database.db

In [ ]:
# Select top purchasing customer for each store based on total purchase amount
# columns: store_id, customer_id, purchase_amt
# tbl: sales
# Input:

store_id, customer_id, purchase_Amt
1, 101, 500
1, 101, 300
1, 102, 500
2, 101, 1000
2, 102, 900
3, 101, 1000
3, 102, 1000


In [ ]:
%%sql
CREATE TABLE sales (
store_id INT,
customer_id INT,
purchase_Amt INT
);

insert into sales (store_id, customer_id, purchase_Amt) 
values(1, 101, 500),
(1, 101, 300),
(1, 102, 500),
(2, 101, 1000),
(2, 102, 900),
(3, 101, 1000),
(3, 102, 1000);


In [ ]:
%%sql 
SELECT * FROM sales;

In [ ]:
%%sql
select store_id, customer_id, SUM(purchase_Amt) as total_purchase,  
DENSE_RANK() OVER(PARTITION BY store_id order by SUM(purchase_Amt) desc) as rank 
from sales group by 1,2;

In [ ]:
%%sql
select store_id, customer_id, total_purchase from (
select store_id, customer_id, total_purchase, DENSE_RANK() OVER(PARTITION BY store_id order by total_purchase desc) as rank from
(select store_id, customer_id, SUM(purchase_Amt) as total_purchase from sales group by 1,2))
where rank=1;

##create pivot table from below table

In [ ]:
##create pivot table from below table
Product		Comp   		Price
iPhone		Amazon		100
iPhone		Flipkart   	90
iPhone		Snapdeal	110
iPhone 11	Amazon		100
iPhone 11	Flipkart   	120
iPhone 11	Snapdeal	110

#expected output
Product   	Amz_price	Flipkart_price	Snap_price
iPhone		100			90				110		
iPhone 11   100			120				110

In [ ]:
%%sql
CREATE TABLE ProductPrice (
    Product VARCHAR(50),
    Comp VARCHAR(50),
    Price INT
);
INSERT INTO ProductPrice (Product, Comp, Price) VALUES
('iPhone', 'Amazon', 100),
('iPhone', 'Flipkart', 90),
('iPhone', 'Snapdeal', 110),
('iPhone 11', 'Amazon', 100),
('iPhone 11', 'Flipkart', 120),
('iPhone 11', 'Snapdeal', 110);

In [ ]:
%%sql
select * from ProductPrice;

In [ ]:
%%sql
select product, 
MAX(CASE WHEN Comp='Amazon' then price else null end) as amazon_price,
MAX(CASE WHEN Comp='Flipkart' then price else null end) as flipkart_price,
MAX(CASE WHEN Comp='Snapdeal' then price else null end) as snapdeal_price
 from ProductPrice
 group by 1;


In [ ]:
## below are two tables
## what is output of below query

sum(a),sum(b),sum(a+b),sum(a)+sum(b)


a	b
1	null
null 1
1	 1

In [3]:
%%sql
CREATE TABLE t1 (a,b);

insert into t1 (a, b) 
values(1,null ),
(null, 1),
(1, 1);

 * sqlite:///my_database.db
(sqlite3.OperationalError) table t1 already exists
[SQL: CREATE TABLE t1 (a,b);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [4]:
%%sql
select * from t1;

 * sqlite:///my_database.db
Done.


a,b
1,None
None,1
1,1
1,None
1,None
2,None
3,None
5,None
None,None


In [2]:
%%sql
select sum(a),sum(b),sum(a+b),sum(a)+sum(b) from t1;

 * sqlite:///my_database.db
Done.


sum(a),sum(b),sum(a+b),sum(a)+sum(b)
14,2,2,16


#### Join results of all joins

In [ ]:
t1  
1
1
2
3
5
Null

t2
1
1
2
4
Null
Null

In [ ]:
%%sql
# CREATE TABLE t2 (a);
insert into t2 (a) 
values(1),
(1),
(2),
(3),
(5),
(NULL);

In [ ]:
%%sql
CREATE TABLE t3 (a);
insert into t3 (a) 
values(1),
(1),
(2),
(4),
(NULL),
(NULL);

In [5]:
%%sql
select * from t2 left join t3
on t2.a=t3.a;

 * sqlite:///my_database.db
Done.


a,a_1
1,1
1,1
1,1
1,1
2,2
3,None
5,None
None,None


In [6]:
%%sql
select * from t2 inner join t3
on t2.a=t3.a;

 * sqlite:///my_database.db
Done.


a,a_1
1,1
1,1
1,1
1,1
2,2


In [7]:
%%sql
select * from t3 left join t2
on t2.a=t3.a;

 * sqlite:///my_database.db
Done.


a,a_1
1,1
1,1
1,1
1,1
2,2
4,None
None,None
None,None


In [8]:
%%sql
select * from t2 full outer join t3
on t2.a=t3.a;

 * sqlite:///my_database.db
(sqlite3.OperationalError) RIGHT and FULL OUTER JOINs are not currently supported
[SQL: select * from t2 full outer join t3
on t2.a=t3.a;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [ ]:
# write query to find net amount per customer ID from below table  
# subtract for debit and add for credit
Id item amount category 
1 a 10 debit 
1 a 20 credit 
1 a 24 debit 
2 b 5 debit 
2 b 50 credit

In [ ]:
%%sql
CREATE TABLE transactions (
    id INT,
    item VARCHAR(50),
    amount INT,
    category VARCHAR(20)
);
INSERT INTO transactions (id, item, amount, category) VALUES
(1, 'a', 10, 'debit'),
(1, 'a', 20, 'credit'),
(1, 'a', 24, 'debit'),
(2, 'b', 5, 'debit'),
(2, 'b', 50, 'credit');


In [9]:
%%sql
select item, 
SUM(CASE when category ='debit' then amount*-1
    when category ='credit' then amount
    ELSE 0
    END) net_amount
from transactions
group by item;

 * sqlite:///my_database.db
Done.


item,net_amount
a,-14
b,45


In [11]:
%%sql
select CAST(tx_date as DATETIME) as dates from cust_transactions;

 * sqlite:///my_database.db
Done.


dates
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024


In [ ]:
#write sql query for pivot
# input table
Product Year Revenue 
A 2021 100 
A 2022 150 
B 2021 200 
B 2022 250

# output table
Product 2021 2022 
A 100 150 
B 200 250

In [12]:
%%sql
select product,
SUM(CASE when year=2021 then revenue END) as y2021,
SUM(CASE when year=2022 then revenue END) as y2022
from product_rev
group by 1;

 * sqlite:///my_database.db
Done.


Product,y2021,y2022
A,100,150
B,200,250


In [13]:
%%sql
select product, year, SUM(revenue) from product_rev
 group by product , year


 * sqlite:///my_database.db
Done.


Product,Year,SUM(revenue)
A,2021,100
A,2022,150
B,2021,200
B,2022,250


In [ ]:
%%sql
-- create example table
CREATE TABLE emp_salary (
    emp_id INTEGER,
    department TEXT,
    salary INT
);

-- sample data (includes ties and a dept with only one employee)
INSERT INTO emp_salary (emp_id, department, salary) VALUES
(1, 'HR', 5000),
(2, 'HR', 7000),
(3, 'HR', 7000),
(4, 'HR', 4000),
(5, 'Eng', 12000),
(6, 'Eng', 11000),
(7, 'Eng', 12000),
(8, 'Eng', 9000),
(9, 'Sales', 8000),
(10,'Sales', 8000),
(11,'Sales', 7000),
(12,'Legal', 6000);

In [ ]:
%%sql
SELECT department, salary AS second_highest_salary
FROM (
    SELECT department, salary,
                 DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rnk
    FROM emp_salary
) t
WHERE rnk = 2
ORDER BY department;

In [ ]:
%%sql
select a, COALESCE(a, 0) from t1;

## what will be output for below tables with joins
Table A
column_a
1
1
1
1
 
Table B     
column_b                            
1
1


left join:  8 rows
right join 8 rows
inner join 8 rows
ourter join 8 rows

In [ ]:
%%sql
# create table A (a int);
# create table B (b int);
# insert into A (a) values(1),(1),(1),(1);
# insert into B (b) values(1),(1);


In [14]:
%%sql
select count(*) from A left join B on A.a=B.b;


 * sqlite:///my_database.db
Done.


count(*)
8


In [15]:
%%sql
select count(*) from A inner join B on A.a=B.b;

 * sqlite:///my_database.db
Done.


count(*)
8


In [ ]:
%%sql
CREATE TABLE Test1 (
  Col1 INT
);
INSERT INTO Test1 (Col1) VALUES
(1),
(1),
(1),
(1);
CREATE TABLE Test2 (
  Col1 INT
);

INSERT INTO Test2 (Col1) VALUES
(1),
(1),
(1),
(1),
(NULL);

In [ ]:
%%sql
SELECT COUNT (*) FROM TEST1 CROSS JOIN TEST2; 

In [ ]:
%%sql
SELECT COUNT (*) FROM TEST1 A, TEST2 B WHERE A.COL1 = B.COL1; 


In [ ]:
%%sql
SELECT COUNT (*) FROM TEST2 A LEFT OUTER JOIN TEST1 B ON A.COL1 = B.COL1; 

## Find the store having maximum customers

In [ ]:
%%sql
CREATE TABLE customer (
    id INT,
    name VARCHAR(50),
    address VARCHAR(20)
);
INSERT INTO customer (id, name, address) VALUES
(1,'A',	'X'),
(2,'B',	'Y'),
(3,'C',	'Z'),
(1,'D',	'X');

CREATE TABLE store (
    id INT,
    name VARCHAR(50),
    location VARCHAR(20)
);
INSERT INTO store (id, name, location) VALUES
(1,'P',	'X'),
(2,'Q',	'W'),
(3,'R',	'Z');

In [ ]:
%%sql
SELECT 
    s.id AS store_id,
    s.name AS store_name,
    COUNT(c.id) AS customer_count
FROM store s
LEFT JOIN customer c 
    ON s.location = c.address
GROUP BY s.id, s.name
ORDER BY customer_count DESC
LIMIT 1;

## sql to get the address for which store not available

In [ ]:
%%sql
select distinct address from customer where Address not in (select distinct location from store);

In [ ]:
%%sql
CREATE TABLE staff (
    staff_id INT PRIMARY KEY,
    first_name VARCHAR(50),
    salary INT,
    department VARCHAR(50)
);

INSERT INTO staff (staff_id, first_name, salary, department) VALUES
(1, 'Monika', 10000, 'HR'),
(2, 'Niharika', 8000, 'Admin'),
(3, 'Vishal', 30000, 'HR'),
(4, 'Amitabh', 50000, 'Admin'),
(5, 'Vivek', 50000, 'Admin'),
(6, 'Vipul', 20000, 'Account'),
(7, 'Satish', 7500, 'Account'),
(8, 'Geetika', 9000, 'Admin');

In [16]:
%%sql
select * from staff

 * sqlite:///my_database.db
Done.


staff_id,first_name,salary,department
1,Monika,10000,HR
2,Niharika,8000,Admin
3,Vishal,30000,HR
4,Amitabh,50000,Admin
5,Vivek,50000,Admin
6,Vipul,20000,Account
7,Satish,7500,Account
8,Geetika,9000,Admin


In [ ]:
%%sql
with avg_salary as (select DEPARTMENT, AVG(SALARY) as average_salary from staff group by DEPARTMENT),
joined_table as (select staff.*, avg_salary.average_salary from staff left join avg_salary on staff.DEPARTMENT= avg_salary.DEPARTMENT)
SELECT *, 
(CASE 
WHEN SALARY > average_salary THEN 'Higher than average' 
 WHEN SALARY < average_salary THEN 'Lower than average'
 WHEN SALARY = average_salary THEN 'Average salary'
END) as catagory from joined_table;

In [ ]:
%%sql
WITH avg_salary AS (
    SELECT 
        DEPARTMENT, 
        AVG(SALARY) AS average_salary 
    FROM staff 
    GROUP BY DEPARTMENT
),
joined_table AS (
    SELECT 
        e.*, 
        a.average_salary
    FROM staff e
    LEFT JOIN avg_salary a
        ON e.DEPARTMENT = a.DEPARTMENT
)
SELECT 
    *,
    CASE 
        WHEN SALARY > average_salary THEN 'Higher than average'
        WHEN SALARY < average_salary THEN 'Lower than average'
        WHEN SALARY = average_salary THEN 'Average salary'
    END AS category
FROM joined_table;


In [ ]:
%%sql
SELECT 
    *, 
    AVG(SALARY) OVER (PARTITION BY DEPARTMENT) as average_salary,
    CASE 
        WHEN SALARY > AVG(SALARY) OVER (PARTITION BY DEPARTMENT) THEN 'Higher than average'
        WHEN SALARY < AVG(SALARY) OVER (PARTITION BY DEPARTMENT) THEN 'Lower than average'
        ELSE 'Average salary'
    END AS category
FROM staff order by 1;

In [ ]:
%%sql
select *, DENSE_RANK() OVER(PARTITION BY DEPARTMENT order by salary desc) as rank
from staff 
order by department, rank;

### SQL: Top 3 Product Categories by Quarterly Revenue

In [ ]:
SELECT
  EXTRACT(YEAR FROM order_date) AS year,
  CONCAT('Q', EXTRACT(QUARTER FROM order_date)) AS quarter,
  product_category,
  SUM(revenue) AS total_revenue
FROM `project.dataset.sales`
GROUP BY year, quarter, product_category
QUALIFY
  ROW_NUMBER() OVER(
    PARTITION BY year, quarter
    ORDER BY SUM(revenue) DESC
  ) <= 3
ORDER BY year, quarter, total_revenue DESC;


In [ ]:
%%sql

create table Teachers (
  id int primary key,
  name varchar(16) not null,
  classroom int not null
);

create table Students (
  id int primary key,
  name varchar(16),
  primary_teacher_id int,
  FOREIGN KEY (primary_teacher_id) REFERENCES Teachers(id)
);

create table Assignments (
  id int primary key,
  teacher_id int not null,
  FOREIGN KEY (teacher_id) REFERENCES Teachers(id)
);

create table Grades (
  student_id int,
  assignment_id int,
  grade decimal (4,1),
  PRIMARY KEY (student_id, assignment_id),
  FOREIGN KEY (student_id) REFERENCES Students(id),
  FOREIGN KEY (assignment_id) REFERENCES Assignments(id)
);

/* End schema */

/* Begin test data */
  
insert into Teachers values
  (4, 'Mr. Feeny', 301), (8, 'Mr. Cooper', 260), (30, 'Ms. Finster', 301);

insert into Students values
  (1, 'Bobby', 4), (2, 'Susie', 8), (3, 'Deborah', 8),
  (4, 'Anand', 4), (5, 'Robert', 4), (6, 'Claire', 4),
  (8, 'Petra', 4), (9, 'Bruce', null), (10, 'Andrew', 4),
  (11, 'Kim', 4);
  
insert into Assignments values(1, 4), (2, 8);

insert into Grades values
	(1, 1, 100), (1, 2, 50),
	(2, 1, 100), (2, 2, 100),
	(3, 1, 40), (3, 2, 8),
	(4, 1, 80), (4, 2, 81),
	(5, 1, 30), (5, 2, 60),
	(6, 2, 90),
	(9, 1, 65), (9, 2, 65),
	(10, 1, 85);

/* We want to identify the school's top-performing students. Design a query, prepared statement, or stored procedure that will 
select only those students whose average score (across all assignments) are in the top half of all students, 
and rank them by their score. Exclude those students who have never completed an assignment.

Expected output:

+------------+-----------+
| student_id | avggrade  |
+------------+-----------+
|          2 | 100.00000 |
|          6 |  90.00000 |
|         10 |  85.00000 |
|          4 |  80.50000 |
+------------+-----------+
*/
/* Add your own statements below */

WITH avg_table as (
select student_id, AVG(grade) as avggrade from Grades group by 1),
ranking as (
select student_id,avggrade, DENSE_RANK() OVER(order by avggrade desc) as rn from  avg_table),
half_mark as (select count(*)/2 as half_count from ranking)

select student_id,avggrade  from ranking 
where rn <= (select half_count from half_mark)
;


/*
select Students.id, Students.name, Teachers.name 
from Students
LEFT JOIN Teachers
ON Students.primary_teacher_id = Teachers.id */

In [3]:
%%sql
WITH avg_table as (
select student_id, AVG(grade) as avggrade from Grades group by 1),
ranking as (
select student_id,avggrade, DENSE_RANK() OVER(order by avggrade desc) as rn from  avg_table),
half_mark as (select count(*)/2 as half_count from ranking)

select student_id,avggrade  from ranking 
where rn <= (select half_count from half_mark)
;

 * sqlite:///my_database.db
Done.


student_id,avggrade
2,100.0
6,90.0
10,85.0
4,80.5


###  Question: Find users who have 3 consecutive days of activity.

In [ ]:
%%sql
-- 1. Create the table
CREATE TABLE user_activity (
    user_id INT,
    activity_date DATE
);

-- 2. Insert sample data
INSERT INTO user_activity (user_id, activity_date) VALUES
-- User 1 has a streak of 4 days (2025-10-01 to 2025-10-04)
(1, '2025-10-01'),
(1, '2025-10-02'),
(1, '2025-10-03'),
(1, '2025-10-04'),
-- User 1 also has a separate non-consecutive activity
(1, '2025-10-06'),

-- User 2 has a streak of 3 days (2025-10-10 to 2025-10-12)
(2, '2025-10-10'),
(2, '2025-10-11'),
(2, '2025-10-12'),
-- User 2 also has a duplicate entry for one day to test the DISTINCT requirement
(2, '2025-10-11'),

-- User 3 only has 2 consecutive days (2025-10-15 and 2025-10-16)
(3, '2025-10-15'),
(3, '2025-10-16'),

-- User 4 has scattered activity, no 3-day streak
(4, '2025-10-01'),
(4, '2025-10-03'),
(4, '2025-10-05');


In [6]:
%%sql
SELECT DATE('2025-01-01', '+7 days');

 * sqlite:///my_database.db
Done.


"DATE('2025-01-01', '+7 days')"
2025-01-08


In [12]:
%%sql
SELECT YEAR(DATE('2017-06-15', '1 days'));

 * sqlite:///my_database.db
(sqlite3.OperationalError) no such function: YEAR
[SQL: SELECT YEAR(DATE('2017-06-15', '1 days'));]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [9]:
%%sql
with LaggedActivity as (SELECT
        user_id,
        activity_date,
        LAG(activity_date, 1) OVER (PARTITION BY user_id ORDER BY activity_date) AS prev_date_1,
        LAG(activity_date, 2) OVER (PARTITION BY user_id ORDER BY activity_date) AS prev_date_2
    FROM
        (SELECT DISTINCT user_id, activity_date FROM user_activity) 
                       )
select user_id, activity_date from LaggedActivity
where activity_date = DATE(prev_date_1, '1 days') and activity_date = DATE(prev_date_2, '2 days')

 * sqlite:///my_database.db
Done.


user_id,activity_date
1,2025-10-03
1,2025-10-04
2,2025-10-12


In [5]:
%%sql
SELECT
        user_id,
        activity_date,
        LAG(activity_date, 1) OVER (PARTITION BY user_id ORDER BY activity_date) AS prev_date_1,
        LAG(activity_date, 2) OVER (PARTITION BY user_id ORDER BY activity_date) AS prev_date_2
    FROM
        (SELECT DISTINCT user_id, activity_date FROM user_activity)


 * sqlite:///my_database.db
Done.


user_id,activity_date,prev_date_1,prev_date_2
1,2025-10-01,None,None
1,2025-10-02,2025-10-01,None
1,2025-10-03,2025-10-02,2025-10-01
1,2025-10-04,2025-10-03,2025-10-02
1,2025-10-06,2025-10-04,2025-10-03
2,2025-10-10,None,None
2,2025-10-11,2025-10-10,None
2,2025-10-12,2025-10-11,2025-10-10
3,2025-10-15,None,None
3,2025-10-16,2025-10-15,None


In [1]:
import pandas as pd
import sqlite3
database_path = 'my_database.db' 
conn =  sqlite3.connect(database_path)

df = pd.read_sql(sql = '''SELECT
        user_id,
        activity_date,
        LAG(activity_date, 1) OVER (PARTITION BY user_id ORDER BY activity_date) AS prev_date_1,
        LAG(activity_date, 2) OVER (PARTITION BY user_id ORDER BY activity_date) AS prev_date_2
    FROM
        (SELECT DISTINCT user_id, activity_date FROM user_activity)''', con=conn
        )

df

,user_id,activity_date,prev_date_1,prev_date_2
0,1,2025-10-01,None,None
1,1,2025-10-02,2025-10-01,None
2,1,2025-10-03,2025-10-02,2025-10-01
3,1,2025-10-04,2025-10-03,2025-10-02
4,1,2025-10-06,2025-10-04,2025-10-03
5,2,2025-10-10,None,None
6,2,2025-10-11,2025-10-10,None
7,2,2025-10-12,2025-10-11,2025-10-10
8,3,2025-10-15,None,None
9,3,2025-10-16,2025-10-15,None


In [2]:
df['activity_date'] = pd.to_datetime(df['activity_date'])

In [3]:
df[df['activity_date'] == pd.to_datetime(df['prev_date_1']) + pd.Timedelta(days=1)][df['activity_date'] == pd.to_datetime(df['prev_date_2'])+ pd.Timedelta(days=2)] 

C:\Users\Prasad\AppData\Local\Temp\ipykernel_10608\1788176109.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[df['activity_date'] == pd.to_datetime(df['prev_date_1']) + pd.Timedelta(days=1)][df['activity_date'] == pd.to_datetime(df['prev_date_2'])+ pd.Timedelta(days=2)]


,user_id,activity_date,prev_date_1,prev_date_2
2,1,2025-10-03,2025-10-02,2025-10-01
3,1,2025-10-04,2025-10-03,2025-10-02
7,2,2025-10-12,2025-10-11,2025-10-10


#### Create Pivot from below table

In [10]:
%%sql
select * from product_rev;

 * sqlite:///my_database.db
Done.


Product,Year,Revenue
A,2021,100
A,2022,150
B,2021,200
B,2022,250


In [16]:
%%sql
select Product,
SUM(CASE when Year = 2021 then Revenue END) as Y2021,
SUM(CASE when Year = 2022 then Revenue END) as Y2022
 from product_rev
 group by 1
;

 * sqlite:///my_database.db
Done.


Product,Y2021,Y2022
A,100,150
B,200,250


##### Q1. Last Order per Customer

Table: orders(customer_id, order_id, order_date, amount)


In [10]:
%%sql
SELECT customer_id, order_id
FROM (
  SELECT
    customer_id,
    order_id,
    ROW_NUMBER() OVER (
      PARTITION BY customer_id
      ORDER BY order_date DESC
    ) AS rn
  FROM orders
)
WHERE rn = 1;

 * sqlite:///my_database.db
(sqlite3.OperationalError) no such table: orders
[SQL: SELECT customer_id, order_id
FROM (
  SELECT
    customer_id,
    order_id,
    ROW_NUMBER() OVER (
      PARTITION BY customer_id
      ORDER BY order_date DESC
    ) AS rn
  FROM orders
)
WHERE rn = 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


#### Write a SQL query to find the top 2 customers by total amount spent in each region over the last 30 days.


In [11]:
%%sql
select * FROM orders;

 * sqlite:///my_database.db
(sqlite3.OperationalError) no such table: orders
[SQL: select * FROM orders;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [4]:
%%sql
SELECT region, customer_id, amount_spent
FROM (
  SELECT
    region,
    customer_id,
    SUM(amount) AS amount_spent,
    RANK() OVER (
      PARTITION BY region
      ORDER BY SUM(amount) DESC
    ) AS rn
  FROM orders
  WHERE order_date >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
  GROUP BY region, customer_id
)
WHERE rn <= 2;

 * sqlite:///my_database.db
(sqlite3.OperationalError) near "(": syntax error
[SQL: SELECT region, customer_id, amount_spent
FROM (
  SELECT
    region,
    customer_id,
    SUM(amount) AS amount_spent,
    RANK() OVER (
      PARTITION BY region
      ORDER BY SUM(amount) DESC
    ) AS rn
  FROM orders
  WHERE order_date >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
  GROUP BY region, customer_id
)
WHERE rn <= 2;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### write sql for below 
input:'20260110' output : '2026-01-10'

In [5]:
%%sql
SELECT
  DATE_FORMAT(STR_TO_DATE('20260110', '%Y%m%d'), '%Y-%m-%d') AS formatted_date;






 * sqlite:///my_database.db
(sqlite3.OperationalError) no such function: STR_TO_DATE
[SQL: SELECT
  DATE_FORMAT(STR_TO_DATE('20260110', '%Y%m%d'), '%Y-%m-%d') AS formatted_date;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
